# Meeting 3: End-to-End Skeleton (Protein Structure Prediction)

> **Cel spotkania:** Implementacja działającego, modularnego potoku (*end-to-end pipeline*) łączącego wejście, model i wyjście.
> Architektura jest w pełni wymienna (*customizable template*) – model dummy lub bazowy szkielet można bez trudu zastąpić docelową architekturą (np. Transformer / ESMFold) w kolejnych etapach.

### Wymagania formalne szablonu predyktora (Meeting 3):
1. **Wejście FASTA:** Wczytanie pliku multi-protein FASTA za pomocą biblioteki `scikit-bio` (`skbio.io.read()`).
2. **Format tensora wejściowego:** Kształt `(B, D, N)`:
   * $B$ – rozmiar batcha (np. 16)
   * $D$ – maksymalna długość sekwencji (np. 1024; krótsze sekwencje są uzupełniane paddingiem, dłuższe ucinane)
   * $N$ – słownik aminokwasów o rozmiarze **22** (20 kanonicznych + `X` nieznany/niestandardowy + `PAD` padding)
3. **Maska reszt:** Tensor `mask` o kształcie `(B, D)` rozróżniający rzeczywiste reszty (`True`) od paddingu (`False`).
4. **Model PyTorch (Template / Dummy):** Przyjmuje `(B, D, N)` oraz `mask`, zwraca przewidziane współrzędne atomów $C_\alpha$.
5. **Wyjście per białko:** Współrzędne o kształcie `(L, 3)`, gdzie $L = \min(\text{długość pierwotna}, D)$.
6. **Format wwPDB v3.3:** Generowanie pliku PDB zawierającego wyłącznie rekordy `ATOM` dla atomów $C_\alpha$, ściśle według specyfikacji wwPDB (dokładnie 1 rekord na przetworzoną resztę, **0 rekordów dla paddingu**, linie po 80 znaków).


In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.predictor import (
    parse_fasta_skbio,
    encode_sequence_one_hot,
    batch_encode_sequences,
    DummyStructurePredictor,
    predict_fasta_to_pdb,
    format_ca_atom_record,
    write_ca_pdb,
    VOCAB_SIZE,
    PAD_IDX,
    UNK_IDX,
    AA_TO_IDX
)

print(f"PyTorch: {torch.__version__}")
print(f"Słownik aminokwasów N = {VOCAB_SIZE} (PAD={PAD_IDX}, UNK={UNK_IDX})")


## Część A: Szablon Predyktora FASTA -> PyTorch (B, D, N) -> wwPDB (L, 3)

### 1. Wczytanie pliku multi-protein FASTA za pomocą `scikit-bio`
Wczytujemy plik `data/samples/sample_multi.fasta` zawierający białka o różnych długościach (krótki peptyd 15 AA, standardowe białko 76 AA, sekwencję z symbolem `X` oraz bardzo długie białko 1200 AA).


In [ ]:
fasta_path = ROOT_DIR / "data/samples/sample_multi.fasta"
records = parse_fasta_skbio(fasta_path)

print(f"Pomyślnie sparsowano {len(records)} sekwencji z pliku FASTA za pomocą scikit-bio:\n")
for pid, desc, seq in records:
    print(f"* ID: {pid:18s} | Długość pierwotna: {len(seq):4d} AA | Opis: {desc}")


### 2. Kodowanie sekwencji do tensora (B, D, N) z maskowaniem (B, D)
Zgodnie ze specyfikacją:
* $B = 4$ (liczba białek w batchu)
* $D = 1024$ (maksymalna długość; krótsze sekwencje są uzupełniane tokenem `PAD`, a sekwencja 1200 AA zostaje obcięta do 1024)
* $N = 22$ (20 aminokwasów + `X` + `PAD`)
* `mask` identyfikuje rzeczywiste reszty (`True`) i pozycje paddingu (`False`).


In [ ]:
sequences = [r[2] for r in records]
D = 1024

x, mask, lengths = batch_encode_sequences(sequences, max_len=D)

print(f"Tensor wejściowy x shape: {x.shape} (wymagany: B={len(sequences)}, D={D}, N={VOCAB_SIZE})")
print(f"Tensor maski mask shape:  {mask.shape} (wymagany: B={len(sequences)}, D={D})")
print(f"Rzeczywiste długości L:   {lengths}")

# Sprawdzenie poprawności maski
for b, (pid, l) in enumerate(zip([r[0] for r in records], lengths)):
    valid_count = mask[b].sum().item()
    pad_count = (~mask[b]).sum().item()
    print(f"  [{pid}] Ważne reszty: {valid_count:4d} | Padded: {pad_count:4d} | Suma: {valid_count + pad_count}")


### 3. Modularny model PyTorch (Dummy Structure Predictor)
Model przyjmuje wejście `(B, D, N)` oraz maskę `(B, D)`. Dla każdego białka zwracany jest tensor o wymiarach `(L, 3)`, gdzie $L = \min(\text{długość}, D)$.


In [ ]:
model = DummyStructurePredictor(vocab_size=VOCAB_SIZE, max_len=D, seed=42)
coords_list = model.predict_coordinates(x, mask=mask, lengths=lengths)

print("Wyjścia modelu po odcięciu paddingu (dokładnie (L, 3) dla każdego białka):")
for b, (pid, coords, l) in enumerate(zip([r[0] for r in records], coords_list, lengths)):
    print(f"  [{pid}] Współrzędne CA shape: {coords.shape} (L={l}, 3 współrzędne X, Y, Z)")
    assert coords.shape == (l, 3)


### 4. Zapis do formatu PDB według specyfikacji wwPDB Format v3.3
Reguły:
* Zapisywane są **wyłącznie rekordy ATOM dla atomów C_alpha**.
* Dokładnie **jeden rekord na przetworzoną resztę**.
* **Zero rekordów dla pozycji paddingu**.
* Ścisły format 80-kolumnowy z terminatorem `TER` i `END`.


In [ ]:
out_dir = ROOT_DIR / "predictions/meeting3_pdb"
pdb_files = predict_fasta_to_pdb(
    fasta_path=fasta_path,
    out_dir=out_dir,
    model=model,
    batch_size=16,
    max_len=D
)

print(f"Wygenerowano {len(pdb_files)} plików PDB w katalogu: {out_dir}\n")
for p in pdb_files:
    with open(p) as f:
        lines = f.readlines()
        atom_lines = [l for l in lines if l.startswith("ATOM  ")]
    print(f"* Plik: {p.name:20s} | Rekordy ATOM: {len(atom_lines):4d} | Długość linii: {len(atom_lines[0].rstrip())} znaków")

print("\nPrzykładowy podgląd pierwszych 5 rekordów PDB (PROT_SHORT.pdb):")
with open(pdb_files[0]) as f:
    for i in range(8):
        print(f.readline().rstrip())


---
## Część B: Szkielet Uczenia i Weryfikacja Przepływu Gradientów

Poniższa część prezentuje szkielet treningowy na danych referencyjnych z AlphaFold DB:
* Wczytanie prawdziwej struktury referencyjnej z binarnego cache lub CIF.
* Różniczkowalna funkcja straty **dRMSD** ($SE(3)$-invariant).
* Sanity check: Overfitting na 1 białku potwierdzający poprawność obliczania gradientów.
* Wykresy macierzy odległości (*contact maps*).


In [ ]:
import os
import sys
from pathlib import Path

# Ustawienie ścieżki do modułów src/
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from src.data import (
    load_split,
    af_id_to_filename,
    parse_cif_ca,
    tokenize_sequence,
    save_coords_to_pdb,
    ProteinStructureDataset,
    collate_protein_batch,
    AA_TO_ID
)
from src.models import ProteinStructureSkeleton, dRMSDLoss
from src.metrics import compute_drmsd, compute_kabsch_rmsd, compute_contact_precision

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")


## 1. Działające Wejście: Przygotowanie Danych i Tokenizacja

Wejściem modelu jest **pierwszorzędowa sekwencja aminokwasowa** (łańcuch liter np. `MGTPCHFAL...`).
Etykietą uczącą (*ground truth*) są współrzędne atomów węgla $\alpha$ ($C_\alpha$) wyciągnięte ze struktur pobranych z bazy **AlphaFold DB**.

Przetestujmy załadowanie przykładowego białka ze zbioru `eval`:


In [ ]:
# Pobieramy ID ze splitu eval
eval_ids = load_split("eval", splits_dir=str(ROOT_DIR / "data/splits"))
sample_id = eval_ids[0]
cif_filename = af_id_to_filename(sample_id)
cif_path = ROOT_DIR / "structures/eval/af" / cif_filename

print(f"Białko: {sample_id}")
print(f"Ścieżka do struktury: {cif_path}")

# Parsowanie sekwencji i współrzędnych CA
sequence, true_coords = parse_cif_ca(str(cif_path))
tokens = tokenize_sequence(sequence)

print(f"Długość sekwencji L = {len(sequence)} reszt")
print(f"Pierwsze 30 aminokwasów: {sequence[:30]}...")
print(f"Pierwsze 30 tokenów:      {tokens[:30]}...")
print(f"Pierwsza współrzędna CA:   {true_coords[0]}")


## 2. PyTorch Dataset & DataLoader z Maskowaniem

Ponieważ białka mają różną długość $L$, nasz `DataLoader` korzysta z dedykowanej funkcji `collate_protein_batch`:
* Dopełnia sekwencje tokenem `<pad>` do maksymalnej długości w danym batchu.
* Generuje tensor logiczny `mask` (`True` dla rzeczywistych reszt, `False` dla paddingu).
* Tworzy macierz rzeczywistych odległości $D_{ij} = \|x_i - x_j\|_2$.


In [ ]:
# Inicjalizacja PyTorch Dataset
dataset = ProteinStructureDataset(
    split="eval",
    splits_dir=str(ROOT_DIR / "data/splits"),
    structures_dir=str(ROOT_DIR / "structures"),
    max_len=256,
    limit=20
)

print(f"Liczba poprawnie zaindeksowanych białek: {len(dataset)}")

# Pobieramy 2 białka i łączymy w batch
batch = collate_protein_batch([dataset[0], dataset[1]])

print("Kształty tensorów w batchu:")
print(f" * tokens:      {batch['tokens'].shape} (Batch, Max_L)")
print(f" * mask:        {batch['mask'].shape} (Batch, Max_L)")
print(f" * true_coords: {batch['coords'].shape} (Batch, Max_L, 3)")
print(f" * true_dists:  {batch['dist_matrix'].shape} (Batch, Max_L, Max_L)")


## 3. Architektura Modelu (ProteinStructureSkeleton)

Nasz model realizuje pełny przepływ od 1D do 3D:
1. **Embedding aminokwasów:** `nn.Embedding(vocab_size=22, embed_dim=64)`
2. **Koder sekwencyjny:** Dwukierunkowa sieć `nn.LSTM(hidden_size=128, bidirectional=True)`, modelująca kontekst lokalny i odległy.
3. **Głowica projekcji 3D:** Warstwy liniowe generujące przesunięcia $\Delta x_i$ w przestrzeni trójwymiarowej.
4. **Fizyczny prior ciągłości łańcucha:** Skalowanie wektorów przesunięcia do $\sim 3.8$ Å (stała odległość między atomami $C_\alpha$ w wiązaniach peptydowych) i kumulacyjna suma współrzędnych.
5. **Różniczkowalna macierz odległości:** Obliczanie odległości euklidesowych pomiędzy wszystkimi parami atomów $D_{ij}$.


In [ ]:
# Inicjalizacja szkieletu modelu
model = ProteinStructureSkeleton(
    embed_dim=64,
    hidden_dim=128,
    num_layers=2,
    predict_mode="displacement"
)

# Forward pass (test przejścia w przód)
model.eval()
with torch.no_grad():
    predictions = model(batch["tokens"], mask=batch["mask"])

pred_coords = predictions["coords"]
pred_dists = predictions["dist_matrix"]

print(f"Wyjściowy tensor współrzędnych 3D: {pred_coords.shape}")
print(f"Wyjściowy tensor macierzy odległości: {pred_dists.shape}")


## 4. Funkcja Straty: dRMSD Loss ($SE(3)$-invariant)

Dlaczego zwykłe MSE na współrzędnych $(x, y, z)$ jest problematyczne?
Jeśli model przewidzi idealny kształt białka, ale obrócony o 90 stopni, MSE będzie ogromne!

Dlatego jako funkcję straty stosujemy **dRMSD (distance-RMSD)**:
$$\mathcal{L}_{dRMSD} = \sqrt{ \frac{1}{L(L-1)} \sum_{i \neq j} (D_{ij}^{pred} - D_{ij}^{true})^2 }$$
Ponieważ macierz odległości $D$ nie zmienia się pod wpływem obrotów ani przesunięć w przestrzeni ($SE(3)$ invariance), model może efektywnie uczyć się fałdowania białka bez konieczności zgadywania jego globalnej orientacji w układzie współrzędnych.


In [ ]:
loss_fn = dRMSDLoss()
initial_loss = loss_fn(pred_dists, batch["dist_matrix"], mask=batch["mask"])
print(f"Początkowa strata dRMSD: {initial_loss.item():.4f} Å")


## 5. Sanity Check: Test Przeuczenia (Overfitting na 1 białku)

Standardowy test poprawności architektury w uczeniu maszynowym:
Jeśli sieć i funkcja straty są zaimplementowane poprawnie, model musi być w stanie **przeuczyć się na pojedynczym przykładzie** (loss powinien gwałtownie spaść w kilkudziesięciu krokach optymalizatora Adam).


In [ ]:
sample = dataset[0]
single_tokens = sample["tokens"].unsqueeze(0)
single_true_dist = sample["dist_matrix"].unsqueeze(0)
single_mask = torch.ones((1, sample["length"]), dtype=torch.bool)

# Optymalizator
optimizer = optim.Adam(model.parameters(), lr=0.01)
model.train()

loss_history = []
print("Rozpoczynanie pętli optymalizacyjnej (60 kroków gradientowych)...")

for step in range(1, 61):
    optimizer.zero_grad()
    out = model(single_tokens, mask=single_mask)
    loss = loss_fn(out["dist_matrix"], single_true_dist, mask=single_mask)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if step % 10 == 0 or step == 1:
        print(f"Krok {step:2d}/60 | dRMSD Loss: {loss.item():.3f} Å")

# Wykres spadku straty
plt.figure(figsize=(7, 3.5))
plt.plot(loss_history, marker='o', markersize=3, color='#1f77b4', lw=2)
plt.title("Sanity Check: Spadek straty dRMSD podczas overfittingu na 1 białku")
plt.xlabel("Krok gradientowy (Step)")
plt.ylabel("dRMSD Loss (Å)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Sukces! Strata spadła z {loss_history[0]:.2f} Å do {loss_history[-1]:.2f} Å.")


## 6. Działające Wyjście (I/O): Eksport do Formatów PDB & Macierzy Kontaktów

Działający szkielet musi generować strukturę w uniwersalnym formacie, który można zinterpretować biologicznie i obejrzeć w trójwymiarze:
1. **Eksport do pliku `.pdb`:** Zapis współrzędnych $C_\alpha$ w standardzie Protein Data Bank.
2. **Macierz odległości i mapa kontaktów:** Porównanie wizualne przewidzianych odległości z referencją AlphaFold.


In [ ]:
# Pobieramy ostateczne predykcje modelu
model.eval()
with torch.no_grad():
    final_out = model(single_tokens, mask=single_mask)
    final_coords = final_out["coords"][0].cpu().numpy()
    final_dists = final_out["dist_matrix"][0].cpu().numpy()

# 1. Zapis do pliku PDB
out_dir = ROOT_DIR / "predictions"
out_dir.mkdir(parents=True, exist_ok=True)
predicted_pdb_path = out_dir / f"{sample['id']}_predicted.pdb"

save_coords_to_pdb(
    coords=final_coords,
    sequence=sample["sequence"],
    out_path=str(predicted_pdb_path)
)
print(f"Przewidziana struktura 3D została zapisana do: {predicted_pdb_path}")

# 2. Obliczenie metryk jakościowych (Meeting 4 preview)
true_coords_np = sample["coords"].numpy()
true_dists_np = sample["dist_matrix"].numpy()

drmsd_val = compute_drmsd(true_coords_np, final_coords)
kabsch_rmsd_val = compute_kabsch_rmsd(true_coords_np, final_coords)
contact_prec = compute_contact_precision(true_coords_np, final_dists, threshold=8.0)

print(f"Metryki ewaluacyjne:")
print(f" * dRMSD:           {drmsd_val:.3f} Å")
print(f" * Kabsch RMSD:     {kabsch_rmsd_val:.3f} Å")
print(f" * Contact Precision (<8Å): {contact_prec * 100:.1f}%")


## 7. Wizualizacja: Porównanie Macierzy Odległości (Contact Maps)

Poniższy wykres zestawia rzeczywistą macierz odległości z modelu AlphaFold z macierzą odtworzoną przez nasz szkielet:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Ground truth
im0 = axes[0].imshow(true_dists_np, cmap="viridis_r", vmin=0, vmax=35)
axes[0].set_title(f"Prawdziwa macierz odległości (AFDB: {sample['id']})")
axes[0].set_xlabel("Indeks reszty")
axes[0].set_ylabel("Indeks reszty")
fig.colorbar(im0, ax=axes[0], label="Odległość (Å)")

# Predykcja szkieletu
im1 = axes[1].imshow(final_dists, cmap="viridis_r", vmin=0, vmax=35)
axes[1].set_title("Przewidziana macierz odległości (Szkielet)")
axes[1].set_xlabel("Indeks reszty")
fig.colorbar(im1, ax=axes[1], label="Odległość (Å)")

plt.tight_layout()
plt.savefig(out_dir / "distance_matrix_comparison.png", dpi=150)
plt.show()


## 8. Podgląd pierwszych linii wygenerowanego pliku PDB


In [ ]:
with open(predicted_pdb_path) as f:
    lines = [f.readline() for _ in range(15)]
print("".join(lines))


## Podsumowanie osiągnięć Meeting 3:
* [x] **Działające Wejście:** Parsowanie sekwencji aminokwasów, tokenizacja i obsługa batchowania z paddingiem.
* [x] **Działający Model:** Sieć neuronowa `ProteinStructureSkeleton` przewidująca współrzędne 3D i macierze dystansów.
* [x] **Działająca Funkcja Straty:** Różniczkowalna strata dRMSD ($SE(3)$-invariant).
* [x] **Weryfikacja Gradientów:** Udany test overfittingu na 1 białku.
* [x] **Działające Wyjście:** Zapis do standardowego formatu `.pdb` i wykresy macierzy kontaktów.
